# 02 · 让代码库像公司的一样：检查、测试、打包、自动化

## 这一章在整个项目里的位置

```
00 账号安全 ✅ → 01 建仓库 ✅ → [02 代码库工程化] → 03 数据进货 → 04–09 Databricks → 10–12 对外服务和 AI
```

前两章搭好了**云上的场地**。这一章搭好**写代码的场地**：之后每一章写的代码，都会自动经过检查和测试，才能进入项目。

## 背景：研究代码和公司代码差在哪

你的 `QAFI_CODE_NEW` 能跑出论文结果，但它有一个公司不能接受的问题：**只有你能跑，只有你知道改了以后有没有坏。**

公司里一个项目有很多人同时改，每天都有新代码进来。靠人记住"这个改动会不会影响别处"是不可能的，所以公司代码库都有一套**自动把关**的流程：

```
你改代码 → 提交前自动检查 → 推到 GitHub → 自动跑全部测试 → 同事审查 → 合并
             (pre-commit)                    (CI)
```

**这一章就是搭这条流程。** 每一步都是公司里的标准做法，面试时问"你们的开发流程是什么样的"，答的就是这个。

## 这一章会做的 7 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 管理依赖 | uv + `pyproject.toml` + `uv.lock` | 换台电脑装出来的包版本不一样 |
| 2 | 代码检查 | ruff | 低级错误、格式不统一 |
| 3 | 测试，包括**黄金测试** | pytest | 改了以后不知道结果变没变 |
| 4 | 提交前把关 | pre-commit | 坏代码、密码、大文件被提交 |
| 5 | 打包成镜像 | Docker | "我电脑上能跑"不等于云上能跑 |
| 6 | 放到 GitHub，自动检查 | GitHub + GitHub Actions（CI） | 每次提交都自动把关 |
| 7 | 让自动检查能访问 AWS | OIDC | 不能把 AWS 密码存进 GitHub |

**花费：0 美元。** 代码文件我都已经写好并验证过，你只需要理解每一步，然后运行那一行命令看结果。

> 遇到不认识的词（Git、CI、Docker、OIDC），先看 [基础知识](../../docs/primer_aws_databricks.md) 第 5 节。

---

## 准备

运行下面这一格。它只是定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = Path.cwd().parents[1] if Path.cwd().name == "A_foundation" else Path.cwd()
os.chdir(ROOT)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 管理依赖：让每台电脑装出一模一样的环境

### 在干什么
把项目需要的 Python 包和它们的**精确版本**写下来，任何人、任何机器都能装出完全一样的环境。

### 背景
你的项目用到 pandas、scikit-learn、boto3 等几十个包，它们每隔几周就出新版本，新版本有时会改变计算结果或者直接报错。研究里常见的情况是："三个月前能跑，今天装了新版本就跑不通了"，或者"我电脑上结果是 0.480，同事电脑上是 0.479"。

### 用了什么技术，为什么
| 文件 / 工具 | 作用 |
|---|---|
| [`pyproject.toml`](../../pyproject.toml) | 项目的"说明书"：需要哪些包（只写大概的版本范围） |
| `uv.lock` | "锁文件"：记录每一个包的**精确版本**，包括包依赖的包 |
| **uv** | 安装工具，按锁文件装，几秒钟就完成 |

`pyproject.toml` 里把包分成两组：
- **运行需要的**（boto3、pandas、scikit-learn）：会被打进第 5 步的镜像，在云上运行
- **只在开发时需要的**（测试、检查工具、notebook）：不会进镜像，镜像更小

**为什么用 uv 而不是 pip 或 conda**：uv 是现在 Python 项目的主流选择，速度快、自带锁文件。conda 在科研里常见，但公司的软件项目里越来越少用。

### 公司里什么情况怎么做
- **要加一个新包**：`uv add 包名`。它会同时更新说明书和锁文件，你把这两个文件一起提交，同事 `uv sync` 一下就有了
- **新同事第一天**：克隆代码，运行 `uv sync`，环境就和所有人一样
- **CI 和 Docker 里**：都用 `uv sync --frozen`，意思是"严格按锁文件装，锁文件和说明书对不上就报错"

### 执行

In [ ]:
run("uv sync")

### 结果怎么看
看到 `Resolved ... packages` 和 `Checked ... packages`，没有报错，就说明环境和锁文件一致。

### 面试怎么说
> "Dependencies are declared in pyproject.toml and pinned in a lock file, so local, CI and Docker all install exactly the same versions."

---

## 第 2 步 · 代码检查：让机器挑低级错误

### 在干什么
用一个工具自动扫描所有代码，找出明显的错误，并把格式统一。

### 背景
研究代码里常见的问题：import 了不用的包、变量名拼错（要跑到那一行才会报错）、每个人缩进和换行习惯不同。在公司里，同事审查你的代码时，不应该把时间花在"这里多了个空格"上，而应该看逻辑对不对。

### 用了什么技术，为什么
**ruff**：一个工具同时做两件事：
- `ruff check`：找问题（未使用的 import、未定义的变量、容易出 bug 的写法）
- `ruff format`：自动统一格式

**为什么是 ruff**：它取代了以前要分开装的 flake8、black、isort 三个工具，快几十倍，现在是 Python 项目的主流选择。规则写在 `pyproject.toml` 的 `[tool.ruff]` 里，全团队共用一套。

### 公司里什么情况怎么做
- **检查报错**：大部分问题 `ruff check --fix` 能自动修；修不了的，按提示改
- **格式不对**：`ruff format` 一键改好
- **团队觉得某条规则不合理**：改 `pyproject.toml` 里的规则，所有人统一生效，而不是各自忽略

### 执行
先检查整个项目，再故意写一段有问题的代码，看它能不能抓到：

In [ ]:
run("uv run ruff check . && uv run ruff format --check .")
Path("src/qafi_platform/_demo.py").write_text("import os\nimport sys\n\n\ndef f(x):\n    return undefined_name + x\n")
run("uv run ruff check src/qafi_platform/_demo.py --output-format concise", check=False)
Path("src/qafi_platform/_demo.py").unlink()

### 结果怎么看
- 整个项目：`All checks passed!`
- 演示代码：`Found 3 errors.`：`os` 和 `sys` 导入了没用（F401），`undefined_name` 没定义（F821）。带 `[*]` 的两个能用 `ruff check --fix` 自动修；最后一个要人来改。这种错误在研究代码里，要等程序真正运行到那一行才会报出来

### 面试怎么说
> "Ruff runs on every commit and in CI, so style and obvious bugs never reach code review."

---

## 第 3 步 · 测试：改了以后，结果还对不对

### 在干什么
写小程序自动检查"代码算出来的结果是不是对的"。其中最重要的是**黄金测试**：用论文的真实数据，重新算出论文里发表的数字。

### 背景
后面每一章都要改这套系统：把科学代码搬进新项目、换存储方式、搬到 Databricks 上训练。每改一次都有可能不小心改坏结果。如果每次都靠人工去对比，迟早会漏。

### 用了什么技术，为什么
**pytest**：Python 的标准测试工具。测试文件放在 `tests/` 里，一条命令全部运行。

我们有两类测试：

| 类型 | 放在哪 | 检查什么 | 例子 |
|---|---|---|---|
| **单元测试** | `tests/unit/` | 一个函数的逻辑对不对，用很小的人造数据 | "相关系数函数：完全正相关时应该接近 1" |
| **黄金测试** | `tests/golden/` | 整套计算能不能重现**已知正确的结果** | 用论文数据算 QAFI 在 Beltran 上的中位相关系数，必须是 **0.480**；在 ClinVar 上的 AUROC 必须是 **0.914** |

**黄金测试为什么重要**：它是这个项目的"保险"。以后任何改动，只要这两个数没变，就说明没改坏。面试里，接手老代码时先写这种测试，叫 **characterization test**，是标准做法。

**黄金数据从哪来**：从论文的原始数据里只抽出需要的几列，压缩成两个小文件（9 MB 和 260 KB），放进代码库。这样 GitHub 的机器上也能重算论文数字。

> 写这一步时有一个小插曲：ClinVar 的 0.914 不是在全部 42,185 个变异上算的，而是在"18 个预测工具都有分数"的 **11,690 个变异**上算的。在全部变异上算只有 0.910。我是用论文实验自己的代码确认的。这类细节正是黄金测试要锁住的东西。

### 公司里什么情况怎么做
- **测试失败**：说明行为变了。先弄清楚是"改坏了"还是"本来就该变"。如果是后者（比如有意改进了算法），要和团队确认后再更新黄金数字，并在提交说明里写清楚原因
- **新写一个功能**：同时写它的单元测试
- **黄金测试的数字永远不随手改**：它是和外部（论文、客户、监管）对齐的结果

### 执行

In [ ]:
run("uv run pytest -v")

### 结果怎么看
全部 `PASSED`。其中 2 个是黄金测试（`test_beltran_median_pearson_qafisplit3` 和 `test_clinvar_auroc_qafisplit3`），其余是单元测试。后面每一章加了新代码都会带上新测试，所以数量会越来越多（比如 `test_raw_validator` 是 03 号的）。跑完要半分钟到一分钟，主要花在读论文数据上。

### 面试怎么说
> "Before refactoring the research code I pinned the paper's published numbers as golden tests — 0.480 on Beltran, 0.914 AUROC on ClinVar — so any change that alters behaviour fails immediately."

---

## 第 4 步 · 提交前把关：坏东西进不了代码库

### 在干什么
设置一组检查，**每次 `git commit` 时自动运行**，不通过就提交不了。

### 背景
有些错误一旦进了代码库，后果很严重：
- **密码或 AWS 密钥被提交**：代码推到 GitHub 上，几分钟内就会被自动扫描的机器人发现并盗用
- **大数据文件被提交**：代码库变得巨大，所有人克隆都要很久，而且几乎删不干净
- **notebook 带着输出被提交**：输出里可能有账号 ID、邮箱、数据

### 用了什么技术，为什么
**pre-commit**：在 `git commit` 那一刻自动运行一组检查。配置写在 [`.pre-commit-config.yaml`](../../.pre-commit-config.yaml)：

| 检查 | 拦住什么 |
|---|---|
| ruff | 代码错误和格式问题（第 2 步） |
| 文件末尾 / 行尾空格 | 无意义的格式差异 |
| 大文件 | 超过 15 MB 的文件（原始数据永远不该进代码库） |
| 私钥 / AWS 密钥 | 密码和密钥 |
| **nbstripout** | **自动清空 notebook 的运行输出** |

**为什么有了 CI 还要它**：CI 在推送之后才运行，而 pre-commit 在提交那一刻就拦住，**密码根本不会离开你的电脑**。

### 公司里什么情况怎么做
- **提交被拦住，提示文件被修改了**：这是工具自动修好了格式。重新 `git add` 再提交一次即可
- **提交被拦住，提示发现密钥**：立刻检查是哪个文件，把密钥移到环境变量或 Secrets Manager。如果密钥已经推送出去过，**马上到 AWS 里作废它**，删提交是不够的

### 执行
先安装：从现在起，每次 `git commit` 都会自动跑这些检查。然后演示一下：假装不小心把一把私钥放进了项目，看它会不会被拦住（演示完马上删掉）：

In [ ]:
run("uv run pre-commit install")
Path("leak_demo.txt").write_text("-----BEGIN " + "RSA PRIVATE KEY-----\nfake-key-for-demo\n")   # 拆成两半写，原因见下面
run("uv run pre-commit run --files leak_demo.txt", check=False)
Path("leak_demo.txt").unlink()

### 结果怎么看
- `pre-commit installed at .git/hooks/pre-commit`：装好了
- 演示文件：`detect private key ... Failed`，并指出 `Private key found: leak_demo.txt`。真提交时只要有一项失败，这次提交就不会发生
- 其他项显示 `Passed` 或 `Skipped`。`Skipped` 表示这一项不管这类文件，比如 ruff 只看 Python 文件

**为什么代码里那句"私钥开头"要拆成两半写**：如果写成完整的一句，这个 notebook 文件本身就包含了"私钥"，第 6 步提交时会被这个检查拦住。我第一次写这一章时就被拦了。它只认内容、不管你是不是演示，这正是它可靠的地方。

真正对全部文件跑一遍是在第 6 步第一次提交的时候。

### 面试怎么说
> "Pre-commit hooks block secrets, large files and notebook outputs at commit time, before anything leaves the laptop."

---

## 第 5 步 · Docker：把程序和环境打成一个包

### 在干什么
把我们的代码和它需要的一切（Python、所有包）打包成一个**镜像**，以后在云上运行的每个任务都从这个镜像启动。

### 背景
"在我电脑上能跑"是软件行业最经典的问题。你的 Mac 和 AWS 的机器操作系统不同、Python 版本可能不同、装的包也可能不同。程序换个地方就出错。

### 用了什么技术，为什么

| 词 | 是什么 | 比喻 |
|---|---|---|
| **镜像**（image） | 打包好的"程序 + 运行环境"，只读 | 外卖盒：饭菜、餐具全在里面，封好了 |
| **容器**（container） | 镜像运行起来的样子 | 打开外卖盒开始吃 |
| **Dockerfile** | 描述镜像怎么做的"菜谱" | 外卖的制作步骤 |

我们的 [`containers/Dockerfile`](../../containers/Dockerfile) 做的事：从一个带 Python 3.12 的基础镜像开始 → 按锁文件装运行需要的包 → 放进我们的代码 → 用普通用户（不是管理员）运行。

**为什么公司离不开它**：本机、CI、AWS 上跑的是**同一个镜像**，环境一致这件事就有了保证。10 号的预测 API 就会用这种镜像。

### 公司里什么情况怎么做
- **改了代码**：重新构建镜像。Dockerfile 里"先装包、再放代码"的顺序是有意的：包没变就用缓存，几秒钟就构建完
- **镜像太大**（我们的 1.5 GB）：用更小的基础镜像、去掉不需要的包。现在够用，以后需要再优化
- **镜像放哪**：推到 AWS 的 **ECR**（镜像仓库），Lambda 和其他服务从那里拉取。10 号会做

### 执行

In [ ]:
run("docker build -q -f containers/Dockerfile -t qafi-platform .")
run("docker run --rm qafi-platform")

### 结果怎么看
- 第一行 `sha256:...` 是新镜像的编号
- 最后一行 `qafi-platform image OK`：镜像能构建、能运行

> 中间那段 `DEPRECATED: The legacy builder is deprecated` 是 Docker 提示有更新的构建工具，不影响结果，可以忽略。
> 如果报错说连不上 Docker：先在终端运行 `colima start`（colima 是你 Mac 上运行 Docker 的引擎）。

### 面试怎么说
> "Everything runs from one Docker image built from the lock file, so the laptop, CI and AWS execute the same environment."

---

## 第 6 步 · 放到 GitHub，每次提交自动检查（CI）

### 在干什么
把项目第一次提交到 Git，推到 GitHub 上的**私有**仓库。之后每次推送，GitHub 自动运行第 2、3、5 步的全部检查。

### 背景
科研里最怕的一句话是"只有在我电脑上能跑"。你的 `QAFI_CODE_NEW` 就是这样：换个人、换台电脑，不一定跑得通。

**CI 就是自动复现**：每次你改完代码，GitHub 自动拿一台全新的电脑（上面没有你电脑里的任何文件），从零把项目装一遍、跑一遍测试。跑通了，0.480 和 0.914 也没变，就证明这个项目谁拿去都能跑。比喻：别的实验室按你的实验记录重做一遍，看能不能得到同样的结果。CI 是每改一次代码，就自动重做一次。

一个例子：你写了新功能，用到一个数据文件，在自己电脑上测试通过了，但**忘了把这个文件提交上去**。pre-commit 看不出少了文件；你自己电脑上跑测试也通过，因为文件就在你电脑上；只有 CI 在全新的电脑上会失败，你马上就知道少交了东西。

公司里还多一层用处：很多人同时提交代码，CI 是**谁都绕不过去的统一检查点**。

### 用了什么技术，为什么

| 词 | 是什么 |
|---|---|
| **Git** | 记录代码每一次修改的工具。每一次"存档"叫一个**提交**（commit） |
| **GitHub** | 存放 Git 仓库的网站，团队在上面协作、审查代码 |
| **CI**（持续集成） | 每次推送代码，自动运行检查和测试 |
| **GitHub Actions** | GitHub 自带的 CI 系统 |
| **Pull Request（PR）** | "请求把我的改动合并进主分支"。CI 结果直接显示在 PR 上 |

我们的 CI 配置在 [`.github/workflows/ci.yml`](../../.github/workflows/ci.yml)，每次推送在 GitHub 的机器上跑 4 项检查：

| 任务 | 检查什么 |
|---|---|
| lint | ruff：代码检查和格式（第 2 步） |
| test | pytest：单元测试 + 黄金测试（第 3 步） |
| terraform | 01 号的 Terraform 代码格式和语法 |
| docker | 镜像能不能构建（第 5 步） |

**为什么公司离不开它**：主分支永远是能跑的。新人提交的代码，在合并之前就已经被自动验证过。

**pre-commit 和 CI 有什么区别，是不是重复了？** 比喻：pre-commit 是**出门前自己检查行李**，CI 是**机场安检**。

| | pre-commit（第 4 步） | CI |
|---|---|---|
| 什么时候跑 | `git commit` 的那一刻 | `git push` 之后 |
| 在哪跑 | 你的电脑上 | GitHub 的干净电脑上 |
| 多快 | 几秒 | 几分钟 |
| 能不能跳过 | 能（`--no-verify`，或者有人根本没装） | 不能，结果所有人都看得到 |
| 查什么 | 快的检查，加上**绝不能离开你电脑的东西**：密码、私钥、大文件、notebook 输出 | 慢的、全面的检查：全部测试、Docker 打包、Terraform 图纸 |

两边都查的只有 ruff。它只要零点几秒，多跑一次不花什么成本：pre-commit 让你早发现，CI 保证谁也绕不过去。密码只能在 pre-commit 查：等 CI 发现，密码已经传到 GitHub 上了，晚了。

第 2、3、5 步手动跑 ruff、pytest、Docker，是为了让你认识每个工具。日常工作里不用手动跑：提交时 pre-commit 自动跑，推送后 CI 自动跑。

### 公司里什么情况怎么做
- **CI 变红**：点进去看是哪个任务失败、日志里的错误，在本地修好再推一次。**红的不合并**，这是规矩
- **为什么本地通过、CI 失败**：通常是忘了提交某个文件，或者依赖没写进锁文件。这正是 CI 存在的意义
- **仓库先建私有**：等项目能展示了，再改成公开放进简历

### 执行
分四小步。

**6a · 提交用哪个邮箱。** 每个提交都会记下作者的邮箱，能看到仓库的人都能看到。你的 git 现在用的是你的 Gmail，而这个仓库以后要公开放进简历。公司里提交用公司邮箱；个人公开仓库的标准做法，是用 GitHub 给每个账号的**隐私邮箱**（`编号+用户名@users.noreply.github.com`），提交照样算在你的 GitHub 账号上。下面的设置只对这个项目生效，不影响你别的项目：

In [ ]:
gh_id = subprocess.run(["gh", "api", "user", "--jq", ".id"], capture_output=True, text=True, check=True).stdout.strip()
run(f"git config user.email {gh_id}+pipi0616@users.noreply.github.com && git config user.email")

**6b · 看清楚要提交什么。** 列出这次会提交的全部文件，再在这些文件里搜一遍你的 Gmail：

In [ ]:
run("git add -A && git status --short")
run('git grep --cached -n -F "$(git config --global user.email)" || echo "文件里没有你的邮箱"', check=False)

确认列表里**没有** `terraform.tfstate`（Terraform 的记录）、`data/`（原始数据）、`.venv/`（虚拟环境），最后一行是"文件里没有你的邮箱"。前三样都被 `.gitignore` 挡在外面了。

**6c · 第一次提交，建 GitHub 仓库，推送。** 这一格会在 GitHub 上**创建一个私有仓库**，所以由你来运行：

In [ ]:
run("uv run pre-commit run", check=False)   # 第一次会自动修一些格式
run("git add -A")
run('git commit -q -m "feat: platform foundation (account baseline, terraform buckets, CI/CD)" && git log --oneline | head -1')
run("gh repo create pipi0616/qafi-ml-platform --private --source=. --remote=origin --push")

**6d · 看 CI 跑。** 推送的那一刻，GitHub 就开始跑 CI 了。在浏览器打开 <https://github.com/pipi0616/qafi-ml-platform/actions>，点进最新的一次运行，能看到任务一个个变绿。两三分钟跑完后，运行下面这一格：

In [ ]:
run("gh run list --limit 3")

### 结果怎么看
- **6c**：第一次运行 pre-commit，`nbstripout` 可能显示 `Failed`、提示 `files were modified by this hook`：它把 notebook 整理了一遍（清掉输出、统一格式）。下一行 `git add -A` 把整理好的版本加进来，提交时会再检查一遍，这次全部 `Passed`。之后会看到一行提交记录（`feat: platform foundation ...`，前面的编号每次不同），最后是仓库地址 `https://github.com/pipi0616/qafi-ml-platform`
- **6d**：最新一行是 `completed  success`。网页上 lint、test、terraform、docker 4 个绿勾；第 5 个任务 `aws` 显示跳过（灰色），第 7 步启用它

> 如果你中途保存过这个 notebook，提交时 nbstripout 也会把它的输出清掉。之后 VS Code 可能提示"文件在磁盘上被修改"，怎么选都行：两边内容一样，区别只是有没有输出。

### 面试怎么说
> "Every push runs lint, unit and golden tests, Terraform validation and a Docker build in GitHub Actions; nothing merges unless it's green."

---

## 第 7 步 · 让 CI 不用密码就能访问 AWS（OIDC）

### 在干什么
让 GitHub 上的自动化流程以后能操作 AWS（上传数据、部署程序），但**不在 GitHub 里存任何 AWS 密码**。

### 背景
最直接的做法是：在 AWS 里生成一对长期密钥，存进 GitHub。这是很多泄露事故的源头：密钥永不过期，GitHub 的任何配置失误、任何有权限的人，都可能让它流出去。

### 用了什么技术，为什么
**OIDC**（OpenID Connect）是一种"用临时证明换临时通行证"的方式：

```
1. GitHub 给这一次 CI 运行签发一张证明：
   "我是 pipi0616/qafi-ml-platform 仓库的一次运行"（几分钟有效）
2. CI 拿这张证明去找 AWS
3. AWS 检查：我信任 GitHub 的签名吗？是我允许的那个仓库吗？
4. 检查通过，AWS 发一张临时通行证（1 小时有效），CI 用它干活
5. 用完自动失效
```

**01 号已经在 AWS 里做好了准备**：告诉 AWS "信任 GitHub 签的证明"，并建了一个角色 `qafi-dev-github-actions`，**只允许 `pipi0616/qafi-ml-platform` 这一个仓库**扮演它。现在这个角色只有只读权限。最小权限：用到什么再加什么。

**为什么是公司标准**：没有任何长期密码存在任何地方；权限精确到"哪个仓库"；每次运行都有记录。

### 公司里什么情况怎么做
- **CI 报 "Not authorized to perform sts:AssumeRoleWithWebIdentity"**：几乎总是仓库名对不上（比如仓库改了名），去改角色的信任条件
- **CI 报权限不够（AccessDenied）**：给角色**只加需要的那一项**权限，不要图省事给管理员权限
- **公司里**：每个仓库、每个环境（dev / prod）一个角色，prod 的角色只允许主分支使用

### 执行
CI 只需要知道角色的名字（ARN）。名字不是密码，所以存成仓库的**变量**（不是机密）：

In [ ]:
run('gh variable set AWS_ROLE_ARN --body "$(terraform -chdir=infra/dev output -raw github_actions_role_arn)" && gh variable list')

设好变量后，CI 的第 5 个任务 `aws` 就会启用。它只做一件事：用 OIDC 登录 AWS，问 AWS"我是谁"。推一个空提交触发它：

In [ ]:
run('git commit --allow-empty -q -m "ci: enable aws oidc job" && git push -q')

同样在 Actions 页面看它跑完（两三分钟），然后运行下面这一格。它从 CI 的日志里找出 AWS 回答的那一行：

In [ ]:
run("gh run view $(gh run list --limit 1 --json databaseId --jq '.[0].databaseId') --log | grep -m1 assumed-role", check=False)

### 结果怎么看
- `gh variable list` 里有 `AWS_ROLE_ARN`，值是 `arn:aws:iam::186972156153:role/qafi-dev-github-actions`
- 最后一格输出里有：

```
"Arn": "arn:aws:sts::186972156153:assumed-role/qafi-dev-github-actions/GitHubActions"
```

`assumed-role` 的意思是：CI 是"扮演角色"进来的，用的是临时通行证，没有密码。网页上这次运行 5 个任务全绿。

> 如果最后一格什么都没输出：多半是 CI 还没跑完，等一会儿再运行一次。

### 面试怎么说
> "CI authenticates to AWS through OIDC: GitHub issues a short-lived token per run, an IAM role trusts it only for this repository, and no long-lived key exists anywhere."

---

## 本章小结

| 步骤 | 技术 | 公司里的作用 |
|---|---|---|
| 1 依赖 | uv、锁文件 | 所有人、所有机器环境一致 |
| 2 检查 | ruff | 低级错误到不了审查 |
| 3 测试 | pytest、**黄金测试** | 改了以后结果不变 |
| 4 提交前把关 | pre-commit | 密码、大文件、notebook 输出进不了代码库 |
| 5 打包 | Docker | 本机 = CI = 云 |
| 6 自动检查 | GitHub Actions | 主分支永远能跑 |
| 7 访问 AWS | OIDC | 不存任何长期密码 |

**一句话**：从这一章起，这个项目的任何改动都会被自动检查，而且论文的两个关键数字被锁住了。

**下一章 03 · 数据进货**：把 86 GB 的全蛋白组压缩包放进 AWS 的 raw 仓库；每个新文件一到，AWS 自动检查它是否完整并登记下来。